# Thick-restarted Arnoldi (Krylov–Schur) on the anb-compute GPU cluster, `alpha_sq = 3`

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

`KrylovSchurLindblad` computes the same quantity as `ArnoldiLindblad` and
`ChebAr` — the eigenvalue of largest real part of the trace-projected one-period
propagator `U`, i.e. the bit-flip rate — but iterates **on `U` itself**
(`n_periods = 1`, `T_block = 2π/ω_a`), restarts every `m = 40` vectors keeping the
`k = 10` Ritz vectors closest to the unit circle, and stops on the **residual
norm** of the target Ritz vector, not on the change of the Ritz value. The final
eigenvalue comes from a Rayleigh quotient on a long block (`U^10`) at tight
integrator tolerance. See `docs/krylov_schur.md`.

This notebook mirrors `notebooks/solvers/arnoldi_no_cheb.ipynb` cell for cell (same
bootstrap, same cluster pattern: one picklable function, plain numpy back).

## What to compare against

`reports/benchmark_methods.md`, `alpha_sq = 3` (quoted there as 6, old
convention), same Hilbert space
(`n_a = 25`, `n_b = 11`, `epsilon_p = 0.1`):

| method | period integrations | rate | res_rel |
|---|---|---|---|
| `arnoldi` (6-period block, m = 90) | 540 | `9.910757e-06` (RQ) | 8.7e-08 |
| `arnoldi_naiv` (1-period, m = 540) | 540 | `9.907906e-06` | 2.0e-09 |
| `cheb_ar` (degree 6, m = 90) | 600 | `9.907873e-06` | 7.5e-09 |

The numbers to look at here are `rate_RQ` (10-block Rayleigh quotient), the
number of period integrations, and the residual. Two runs are made in one job
each: the loop at a relaxed integrator tolerance (`1e-7 / 1e-8`) with tight
polishing cycles at the end, and the loop at the tight tolerance throughout
(`1e-9 / 1e-10`, what the other solvers use).

## Running it

Kernel and token setup are exactly as in `notebooks/solvers/arnoldi_no_cheb.ipynb`
(pixi environment, Python 3.13, the first cell mints the Ray token). Run all
cells. Each job is a couple of minutes of GPU time plus startup.


In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

In [ ]:
def run_krylov_schur(
    n_a=25,
    n_b=12,
    alpha_sq=3.0,
    epsilon_p=0.1,
    # Krylov-Schur
    m=40,
    k=8,
    res_tol=1e-12,
    max_cycles=25,
    # integrator tolerance for the Arnoldi loop ...
    rtol_loop=1e-7,
    atol_loop=1e-8,
    # ... switched to the final tolerance once the residual reaches this, or
    # after `max_cycles_loop` cycles (the relaxed tolerance has a residual
    # floor of its own, which `res_tol_loop` may sit below) ...
    res_tol_loop=1e-6,
    max_cycles_loop=None,
    # ... which is also what the final Rayleigh quotient uses
    rtol_final=1e-12,
    atol_final=1e-13,
    n_blocks_final=100,
    seed=0,
):
    """Thick-restarted Arnoldi on the one-period propagator, on one GPU worker.

    Returns plain numpy/python objects only: the client has neither jax nor
    dynamiqs. The Krylov basis (m+1 vectors of N^2 complex numbers) never
    crosses the wire; per-cycle Ritz values, residuals and timings do, plus the
    target Ritz vector for the Wigner plots.

    Physics constants come from `floquet_lindblad.models.ats` (single source of truth).
    """
    import time

    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad import KrylovSchurLindblad
    from floquet_lindblad.models.ats import build_ats_hamiltonian_interaction_compensated_shift

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)

    # --- interaction frame, ONE drive period per block: the iteration runs on U ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction_compensated_shift(
        n_a=n_a,
        n_b=n_b,
        alpha_sq=alpha_sq,
        epsilon_p=epsilon_p,
        n_periods=1,
    )

    solver = KrylovSchurLindblad(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
        rtol=rtol_loop,
        atol=atol_loop,
    )
    x0 = solver.make_x0(seed=seed)
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]

    # --- final numbers on the true propagator, tight tolerance ---------------
    # One block: residual of the eigenvector. Ten blocks: the Rayleigh quotient
    # that resolves 1 - mu^10 well above the integrator tolerance; this is the
    # rate to compare with the reports.
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks_final, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- lab frame, Wigners, Fock weights (as in test_arnoldi_no_cheb) --------
    V = np.asarray(V)
    rho_lab = V @ np.asarray(solver.unvec(x_ritz)) @ V.conj().T
    rho_a = dq.ptrace(dq.asqarray(rho_lab, dims=dims), 0)
    xvec_a, yvec_a, W_a = dq.wigner(rho_a)
    rho_b = dq.ptrace(dq.asqarray(rho_lab, dims=dims), 1)
    xvec_b, yvec_b, W_b = dq.wigner(rho_b)
    D = np.einsum("ij,ij->i", rho_lab, rho_lab.conj()).real.reshape(n_a, n_b)

    def plain_history(h):
        return {
            "cycle": int(h["cycle"]),
            "n_apply": int(h["n_apply"]),
            "mu": complex(h["mu"]),
            "res_rel": float(h["res_rel"]),
            "theta": np.asarray(h["theta"]),
            "res": np.asarray(h["res"]),
            "theta_kept": None if h["theta_kept"] is None else np.asarray(h["theta_kept"]),
            "defect": h["defect"],
            "t_cycle": float(h["t_cycle"]),
            "rtol": float(h["rtol"]),
            "atol": float(h["atol"]),
        }

    import jax

    return {
        # eigenvalue from the Krylov iteration (one period) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_RQ": py(res_n["rate_RQ"]),
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": n_blocks_final,
        # cost
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "history": [plain_history(h) for h in history],
        "ritz_final": {
            "theta": np.asarray(out["ritz"]["theta"]),
            "res": np.asarray(out["ritz"]["res"]),
            "idx_target": int(out["ritz"]["idx_target"]),
        },
        "log": lines,
        # state for plots / checks
        "x_ritz": np.asarray(x_ritz),
        "rho_a": np.asarray(dq.to_jax(rho_a)),
        "rho_b": np.asarray(dq.to_jax(rho_b)),
        "fock_weight_a": D.sum(axis=1) / D.sum(),
        "fock_weight_b": D.sum(axis=0) / D.sum(),
        "wigner_a": (np.asarray(xvec_a), np.asarray(yvec_a), np.asarray(W_a)),
        "wigner_b": (np.asarray(xvec_b), np.asarray(yvec_b), np.asarray(W_b)),
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": py(T_block),
        "settings": {
            "n_a": n_a,
            "n_b": n_b,
            "alpha_sq": alpha_sq,
            "epsilon_p": epsilon_p,
            "m": m,
            "k": k,
            "res_tol": res_tol,
            "res_tol_loop": res_tol_loop,
            "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop,
            "atol_loop": atol_loop,
            "rtol_final": rtol_final,
            "atol_final": atol_final,
            "max_cycles": max_cycles,
            "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }


In [ ]:
import time

# Two jobs, run sequentially so the timings are comparable. Both iterate on U
# (one drive period) with m = 40, k = 10.
RUNS = {
    "loop 1e-7, polish 1e-9": dict(rtol_loop=1e-7, atol_loop=1e-8)
}#,
  #  "tight 1e-9 throughout": dict(rtol_loop=1e-11, atol_loop=1e-12),
#}

results = {}
for name, kwargs in RUNS.items():
    t0 = time.time()
    results[name] = acr.run(
        run_krylov_schur,
        num_gpus=1,
        num_cpus=4,
        runtime_env=RUNTIME_ENV,
        poll_seconds=5,
        **kwargs,
    )
    r = results[name]
    print(f"\n[{name}] wall clock incl. startup: {time.time() - t0:.1f} s | device {r['device']}")
    print("  timings:", {kk: round(v, 1) for kk, v in r["timings_s"].items()})
    print("\n".join("  " + s for s in r["log"]))


In [ ]:
import numpy as np

RATE_REF = 1.4e-5#1.5033113711491237e-08

print(f"{'run':<26} {'rate_RQ (100 blocks)':>20} {'rel dev':>9} {'rate (Ritz, 1 blk)':>19} "
      f"{'res Krylov':>11} {'res 1blk':>9} {'apps':>5} {'cycles':>6} {'GPU s':>6}")
for name, r in results.items():
    rate = np.real(r["rate_RQ"])
    print(
        f"{name:<26} {rate:20.6e} {abs(rate / RATE_REF - 1):9.1e} "
        f"{np.real(r['rate_ritz']):19.6e} {r['res_rel']:11.2e} {r['res_rel_1']:9.2e} "
        f"{r['n_apply']:5d} {r['n_cycles']:6d} "
        f"{r['timings_s']['loop'] + r['timings_s']['polish']:6.1f}"
    )
print(f"\nreference (arnoldi, 6-period, m=90, 540 apps): {RATE_REF:.6e}")
for name, r in results.items():
    mu = r["mu"]
    print(f"\n[{name}] converged={r['converged']}  mu = {mu:.12f}")
    print(f"  1 - Re mu = {1 - mu.real:.3e} per period, Im mu = {mu.imag:.1e} (noise indicator)")
    print(f"  mu_RQ^(10) = {r['mu_RQ_n']:.12f}, 1 - |mu_RQ^(10)| = {1 - abs(r['mu_RQ_n']):.3e}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Categorical hues in fixed order (see the dataviz palette): blue, orange.
COLORS = ["#2a78d6", "#eb6834"]

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.8), constrained_layout=True)

for (name, r), c in zip(results.items(), COLORS):
    apps = np.array([h["n_apply"] for h in r["history"]])
    res = np.array([h["res_rel"] for h in r["history"]])
    gap = np.array([abs(1 - h["mu"].real) for h in r["history"]])
    axes[0].plot(apps, res, ".-", ms=5, lw=1.2, color=c, label=name)
    axes[1].plot(apps, gap, ".-", ms=5, lw=1.2, color=c, label=name)
    # where the integrator tolerance was tightened
    tol = np.array([h["rtol"] for h in r["history"]])
    switch = np.flatnonzero(np.diff(tol) != 0)
    for s_ in switch:
        axes[0].axvline(apps[s_ + 1], color=c, lw=0.8, ls=":")

axes[0].axhline(results[next(iter(results))]["settings"]["res_tol"], color="0.4", lw=0.8, ls="--",
                label="res_tol")
axes[0].set_yscale("log")
axes[0].set_xlabel("period integrations")
axes[0].set_ylabel(r"residual $\|U y - \theta y\| / |\theta|$")
axes[0].set_title("target residual per cycle (dotted: tolerance tightened)", fontsize=9)

axes[1].set_yscale("log")
axes[1].set_xlabel("period integrations")
axes[1].set_ylabel(r"$1 - \mathrm{Re}\,\theta$  (per period)")
axes[1].set_title("target Ritz value per cycle", fontsize=9)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle(rf"Krylov–Schur on $U$, $|\alpha|^2 = {r['settings']['alpha_sq']}$, "
             rf"$m = {r['settings']['m']}$, $k = {r['settings']['k']}$", fontsize=10)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ritz spectrum of the final cycle of the first run: all Ritz values, the
# target, and the k values a further restart would keep (target + largest
# moduli). The right panel zooms on the slow end of the real axis.
name = next(iter(results))
r = results[name]
theta = r["ritz_final"]["theta"]
res = r["ritz_final"]["res"]
idx_t = r["ritz_final"]["idx_target"]
k = r["settings"]["k"]
order = [idx_t] + [i for i in np.argsort(-np.abs(theta)) if i != idx_t]
kept = np.array(order[:k])
mask_kept = np.zeros(len(theta), bool)
mask_kept[kept] = True

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.4), constrained_layout=True)
phi = np.linspace(0, 2 * np.pi, 400)
for ax in axes:
    ax.plot(np.cos(phi), np.sin(phi), color="0.6", lw=0.8)
    ax.scatter(theta.real[~mask_kept], theta.imag[~mask_kept], s=14, color="0.55",
               label="discarded at a restart")
    ax.scatter(theta.real[mask_kept], theta.imag[mask_kept], s=22, color="#2a78d6",
               label=f"kept (k = {k})")
    ax.scatter([theta.real[idx_t]], [theta.imag[idx_t]], s=70, marker="*",
               color="#eb6834", zorder=3, label="target (largest Re)")
    ax.set_xlabel(r"$\mathrm{Re}\,\theta$")
    ax.set_ylabel(r"$\mathrm{Im}\,\theta$")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, lw=0.4, color="0.9")
    ax.set_axisbelow(True)

axes[0].set_aspect("equal")
axes[0].set_xlim(-1.1, 1.1)
axes[0].set_ylim(-1.1, 1.1)
axes[0].set_title("all Ritz values, final cycle", fontsize=9)
axes[0].legend(fontsize=7, frameon=False, loc="lower left")

slow = theta.real > 0.8
axes[1].set_xlim(0.8, 1.001)
im_max = max(1e-3, 1.2 * np.abs(theta.imag[slow]).max()) if slow.any() else 1e-2
axes[1].set_ylim(-im_max, im_max)
axes[1].set_title("slow end (Re > 0.8); residual of each shown as text", fontsize=9)
for i in np.flatnonzero(slow):
    axes[1].annotate(f"{res[i]:.0e}", (theta.real[i], theta.imag[i]),
                     textcoords="offset points", xytext=(4, 3), fontsize=6, color="0.3")
fig.suptitle(f"[{name}] Ritz spectrum of the one-period propagator", fontsize=10)
plt.show()


In [ ]:
# --- Fock-truncation check (same criterion as notebooks/solvers/arnoldi_no_cheb.ipynb) ---
#
# The weight in the top K_TOP levels must be below TOL: necessary, not
# sufficient, and it has to land far below 1 - |mu| and below res_rel.

K_TOP = 3
TOL = 1e-11

def check_mode(label, w, rho, lam_ref, k_top=K_TOP, tol=TOL):
    w = np.asarray(w)
    diag = np.real(np.diag(np.asarray(rho)))
    n = len(w)
    tail = float(w[n - k_top :].sum())
    passed = tail < tol
    print(f"mode {label}: cutoff = {n}, reference lambda = {lam_ref:.3g}")
    print(f"  {'n':>4} {'weight w[n]':>13} {'rho[n,n]':>13}")
    for kk in range(n - k_top, n):
        print(f"  {kk:4d} {w[kk]:13.3e} {diag[kk]:13.2e}")
    print(f"  top-{k_top} weight = {tail:.3e}")
    print(f"  tol = {tol:.0e}  ->  {'PASS' if passed else 'FAIL'}")
    return passed


for name, r in results.items():
    s, p = r["settings"], r["params"]
    lam_b = (p["epsilon_d"] / p["kappa_b"]) ** 2
    print(f"=== {name} ===")
    print(f"res_rel (1 block, tight) = {r['res_rel_1']:.2e}   1 - |mu| = {1 - abs(r['mu']):.2e}\n")
    ok_a = check_mode("a (storage)", r["fock_weight_a"], r["rho_a"], s["alpha_sq"])
    print()
    ok_b = check_mode("b (buffer)", r["fock_weight_b"], r["rho_b"], lam_b)
    print("\nTRUNCATION CHECK:", "PASSED" if ok_a and ok_b else "FAILED", "\n")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Memory-mode Wigner function of the target Ritz vector, both runs. The
# bit-flip mode is two lobes of opposite sign without interference fringes;
# fringes without lobes would mean a phase-flip coherence was selected.
vmax = 2 / np.pi
fig, axes = plt.subplots(1, len(results), figsize=(5.0 * len(results), 4.2),
                         constrained_layout=True)
for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):
    xvec, yvec, W = r["wigner_a"]
    im = ax.pcolormesh(
        xvec, yvec, np.clip(W.T, -vmax, vmax),
        cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
    )
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
    ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
    ax.set_title(f"memory mode, {name}", fontsize=9)
    fig.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
plt.show()
